# LS-NEQ 08_compass_heads — Domänen-Kompass, Domänenachsen und Heads

**Dossier-Stufe I** (`lsne.experiments.STAGES["I"]`, siehe `docs/DESIGN.md` §8 und `docs/STAGES.md`).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Wohin steuern? (`docs/DESIGN.md` §4.7) Domänenprofil des State-2143-Zustands h₂₀₇ und der Richtungen `ent`, `lang_dom`, `freq` (Logit-Lens-Proxy des Suszeptibilitätsatlas: beförderte Masse pro Domäne, z gegen Zufallsrichtungen gleicher Dosis), Import der gespeicherten Atlas-Extraktion (optional), Domänenachsen `dom_law`, `dom_culture_politics`, `dom_protein_misfolding` (Difference-of-Means auf Domänensätzen), **Gegensteuern** (Jura-Anteil herausprojizieren, dann Kultur/Politik-Dosis, als Substitution geschrieben) gegen `dom_law`, `lang_dom` und Zufall nach Familiendomäne, Head-Attribution (Mean-Ablation, direkter Logit-Beitrag) und Steering am **schlechtesten Head** vs. am Residualstrom.

**Eingaben.** Stimulusbank (für den Domänenvergleich die Familien F13–F16 Kultur/Politik, F17–F18 Jura, F19–F20 Proteinfehlfaltung, Rest Alltag), Richtungsbank, Domänenlexikon `data/lexicon/domains.json`, optional `data/atlas/*.txt` (ATLAS_DIR), Anker für h₂₀₇.

**Ausgaben.** Tabellen `compass`, `compass_summary`, `atlas_overlap`, `domain_axes`, `direction_domain`, `direction_domain_T`, `direction_domain_boot`, `head_effects`, `head_direct`, `worst_head`, `head_steering`, `head_steering_boot`; Achsen in `directions_domain.npz`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/I/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/08_compass_heads/`; am Ende eine kurze `summary_08_compass_heads_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Stufe-I-Parameter → Modell → Stufe I → Domänenprofile → Domänenachsen → Richtung × Domäne → Head-Attribution und schlechtester Head → Head-Arm → Zusammenfassung.

**Präregistrierung.** Kontrast 12 (`compass_summary`, `direction = ent`, `lang = E`: `law_is_top` / `law_rank` — Dokumentation, kein Test), Kontrast 13 (`direction_domain_boot`, `quantity = B_cs_minus_lang_dom`, `family_domain = culture_politics`, K=8/C1, Mindesteffekt 0,05 nats) und Kontrast 14 (`head_steering_boot`, `quantity = B_head_minus_residual`, zweiseitig; Head-Auswahl auf Entwicklungsfamilien eingefroren).

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
import os

try:
    from google.colab import drive            # type: ignore
    drive.mount("/content/drive")
except Exception as e:                        # not on Colab, or the mount was refused
    print("Kein Colab-Drive:", repr(e).split("(")[0], "-> lokale Ausgabe")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "08_compass_heads"
STAGE = "I"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Stufe-I-Parameter ---------------------------------------------------------------
ATLAS_DIR = None   # Verzeichnis der gespeicherten Atlas-Extraktion (*.txt / *.md); None = data/atlas; z. B. "/content/drive/MyDrive/LSNE/inputs/atlas"
_atlas = ATLAS_DIR or str(C._REPO_ROOT / "data" / "atlas")
_n_atlas = len([f for f in (os.listdir(_atlas) if os.path.isdir(_atlas) else []) if f.endswith((".txt", ".md"))])
print("Domänen-Lexikon:", C._REPO_ROOT / "data" / "lexicon" / "domains.json")
print(f"Atlas-Verzeichnis: {_atlas} ({_n_atlas} Datei(en); ohne Dateien läuft nur der Logit-Lens-Proxy)")
print(f"Dosen B(x): {limits.i_doses}; Head-Arm-Dosen (relativ zur Head-RMS): {limits.i_head_eps}; "
      f"Zufallsrichtungen des Domänenprofils: {limits.i_n_random_profile}")
if ATLAS_DIR:
    CLI_COMMAND += f" --atlas-dir {ATLAS_DIR}"


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(status="reused", run_dir=str(rd.path), seconds=0.0, error="")
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(status="ok", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="")
        print(f"Stufe {stage} ok in {report[stage]['seconds']:.0f} s: {', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Substufen-Bericht: die Stufen G-L fangen Fehler pro Substufe ab (docs/STAGES.md) ----
# A failed substage is recorded in manifest.json -> substages.errors (traceback in
# substage_errors.txt), its table stays empty and the stage goes on; the manifest's
# `prereg` map names the table / rows that carry every pre-registered contrast.
def substage_report(stage: str) -> dict:
    """Print the substage report and the pre-registration map of a G-L manifest."""
    rd = RunDir(cfg.out_root, stage, cfg.model)
    if not rd.manifest_path.exists():
        print(f"Stufe {stage}: kein manifest.json in {rd.path}")
        return {}
    m = read_manifest(rd)
    sub = m.get("substages") or {}
    rt = m.get("runtime_s")
    print(f"Stufe {stage}: Status {m.get('status')}" + (f", {rt:.0f} s" if isinstance(rt, (int, float)) else "")
          + f", Substufen fehlgeschlagen: {sub.get('n_failed', '?')}"
          + (" | STUB: lsne.transplant nicht importierbar" if m.get("stub") else ""))
    for name, err in (sub.get("errors") or {}).items():
        print(f"  FEHLER in Substufe {name}: {err}")
    if sub.get("n_failed"):
        print(f"  Tracebacks: {rd.path / 'substage_errors.txt'}")
    for k, v in (m.get("prereg") or {}).items():
        print(f"  Präregistrierung {k}: {v}")
    return m


def truthy(s: pd.Series) -> pd.Series:
    """Boolean view of a flag column (bool in memory, 'True'/'False' strings after a csv reload)."""
    return s.astype(str).str.lower().eq("true")


In [ ]:
# --- Stufe I ausführen ----------------------------------------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage I --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT> [--atlas-dir data/atlas]
res_I = run_stage("I", atlas_dir=ATLAS_DIR)
print({k: len(v) for k, v in res_I.items()})
manifest_I = substage_report("I")
if manifest_I:
    print("Zustand h₂₀₇ aus:", manifest_I.get("state_source"), "| schlechtester Head:", manifest_I.get("worst_head"),
          "| verworfene Lexikonwörter (mehr als ein Token):", manifest_I.get("domain_ids_dropped"))


In [ ]:
# --- Kontrast 12: Domänenprofil von h₂₀₇ und der Richtungen (z gegen Zufallsrichtungen) ----------
with guarded("Domänen-Kompass"):
    comp = table("I", "compass")
    summ = table("I", "compass_summary")
    print("Zustand:", comp["state_source"].iloc[0] if len(comp) else "?", "| Atlas-Erwartung für ent: law (Jura) ganz oben")
    show(summ[summ["state_source"] != "sae_decoder"], n=40)
    lang0 = "E"
    sub = comp[(comp["lang"] == lang0) & (comp["state_source"] != "sae_decoder")]
    dirs = [d for d in ["state", "ent", "lang_dom", "freq", "dom_law", "dom_culture_politics", "dom_protein_misfolding"]
            if d in set(sub["direction"])]
    doms = sorted(set(sub["domain"]))
    piv = sub.pivot_table(index="direction", columns="domain", values="z").reindex(dirs)[doms]
    fig, ax = plt.subplots(figsize=(0.8 * len(doms) + 3, 0.5 * len(dirs) + 2))
    vmax = float(np.nanmax(np.abs(piv.to_numpy(dtype=float)))) or 1.0
    im = ax.imshow(piv.to_numpy(dtype=float), cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(doms)))
    ax.set_xticklabels(doms, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(dirs)))
    ax.set_yticklabels(dirs)
    for i in range(len(dirs)):
        for j in range(len(doms)):
            val = piv.iat[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:+.1f}", ha="center", va="center", fontsize=7)
    ax.set_title(f"Domänenprofil (Lexikon {lang0}): z der beförderten Masse pro Domäne, ε=0,05, "
                 f"gegen {limits.i_n_random_profile} Zufallsrichtungen", fontsize=9)
    fig.colorbar(im, ax=ax, label="z", shrink=0.8)
    savefig("compass_domain_profile")
    plt.show()
    atl = table("I", "atlas_overlap")
    if len(atl):
        print("Überlappung der Atlas-Cluster (Schlüsselwörter) mit den Domänenlexika:")
        show(atl.sort_values("jaccard", ascending=False), n=30)
    else:
        print("Keine Atlas-Extraktion gefunden (data/atlas bzw. ATLAS_DIR ohne *.txt) -> nur der lokale Proxy.")


In [ ]:
# --- Domänenachsen: Kosinus zur Richtungsbank und untereinander --------------------------
with guarded("Domänenachsen"):
    ax_t = table("I", "domain_axes")
    show(ax_t)
    cos_cols = [c for c in ax_t.columns if c.startswith("cos_") and c != "cos_denoised_plain"]
    M = ax_t.set_index("name")[cos_cols].astype(float)
    fig, ax = plt.subplots(figsize=(0.9 * len(cos_cols) + 2, 2.6))
    im = ax.imshow(M.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(cos_cols)))
    ax.set_xticklabels([c[4:] for c in cos_cols], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(M)))
    ax.set_yticklabels(M.index, fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M.iat[i, j]):
                ax.text(j, i, f"{M.iat[i, j]:+.2f}", ha="center", va="center", fontsize=7)
    ax.set_title("Kosinus der Domänenachsen zu ent / lang_dom / freq / lang_probe und untereinander", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8)
    savefig("domain_axes_cosines")
    plt.show()


In [ ]:
# --- Kontrast 13: B(x) pro Richtung × Familiendomäne (Gegensteuern gegen dom_law, dann dom_culture_politics) --
with guarded("Richtung × Domäne"):
    dd = table("I", "direction_domain")
    db = table("I", "direction_domain_boot")
    std = dd[dd["kind"] == "standard"] if "kind" in dd else dd
    eps0 = 0.05 if np.isclose(std["eps"], 0.05).any() else float(sorted(set(std["eps"]))[-1])
    sub = std[np.isclose(std["eps"], eps0)]
    sub = sub[sub["K"] == 8] if (sub["K"] == 8).any() else sub
    order = [d for d in ["dom_culture_politics_cs", "dom_culture_politics", "dom_law", "lang_dom", "rand"] if d in set(sub["direction"])]
    groups = list(dict.fromkeys(sub["family_domain"]))
    piv = sub.groupby(["family_domain", "direction"])["B"].mean().unstack("direction").reindex(groups)[order]
    ax = piv.plot(kind="bar", figsize=(8, 4), width=0.8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel(f"B = ln p_δ − ln p₀ [nats], ε={eps0:g}, K=8/C1")
    ax.set_xlabel("Familiendomäne")
    ax.set_title("Ist Steuern in der Schwachdomäne (Kultur/Politik) treffsicherer? (cs = nach Gegensteuern)", fontsize=10)
    ax.legend(fontsize=7, ncol=2)
    savefig("direction_x_domain")
    plt.show()
    q = db[db["quantity"] == "B_cs_minus_lang_dom"]
    if len(q):
        print("Präregistrierter Kontrast 13: B(dom_culture_politics nach Gegensteuern) − B(lang_dom), Familien-Bootstrap, Mindesteffekt 0,05 nats:")
        show(q[["family_domain", "K", "C", "eps", "preregistered", "min_effect_met", "mean", "ci_low", "ci_high", "n_clusters"]])
        if not truthy(q["preregistered"]).any():
            print("Hinweis: die präregistrierte Zeile braucht Kultur/Politik-Familien (F13–F16) im Lauf (FAMILIES = None).")
    T = table("I", "direction_domain_T")
    if len(T):
        print("Texteffekt T(K=8, C1) pro Familiendomäne:")
        show(T.groupby("family_domain")["T"].agg(["mean", "std", "count"]))


In [ ]:
# --- Head-Attribution: Mean-Ablation und direkter Logit-Beitrag pro Head, schlechtester Head ----
with guarded("Head-Attribution"):
    eff = table("I", "head_effects")
    wh = table("I", "worst_head")
    hd = table("I", "head_direct")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.8))
    piv = eff.groupby(["layer", "head"])["delta"].mean().unstack("head")
    vmax = float(np.nanmax(np.abs(piv.to_numpy(dtype=float)))) or 1.0
    im = a1.imshow(piv.to_numpy(dtype=float), cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    a1.set_xticks(range(piv.shape[1]))
    a1.set_xticklabels(piv.columns)
    a1.set_yticks(range(piv.shape[0]))
    a1.set_yticklabels([f"L{l}" for l in piv.index])
    a1.set_xlabel("Head")
    a1.set_title("Δ ln p(Ziel) bei Mean-Ablation (Mittel über Varianten; Stern = schlechtester Head)", fontsize=9)
    fig.colorbar(im, ax=a1, shrink=0.8)
    if len(wh):
        sel = wh[truthy(wh["selected"])].iloc[0]
        a1.plot([list(piv.columns).index(sel["head"])], [list(piv.index).index(sel["layer"])], marker="*", color="k", ms=14)
        print(f"Schlechtester Head: L{sel['layer']}H{sel['head']} (Auswahl: {sel['selection']}; schwache Familien: "
              f"{sel['families_weak']}; stabile Familien: {sel['families_stable'] or '-'})")
        show(wh.head(8))
    pv2 = hd.groupby(["layer", "head"])["direct_effect"].mean().unstack("head")
    vmax2 = float(np.nanmax(np.abs(pv2.to_numpy(dtype=float)))) or 1.0
    im2 = a2.imshow(pv2.to_numpy(dtype=float), cmap="RdBu_r", vmin=-vmax2, vmax=vmax2, aspect="auto")
    a2.set_xticks(range(pv2.shape[1]))
    a2.set_xticklabels(pv2.columns)
    a2.set_yticks(range(pv2.shape[0]))
    a2.set_yticklabels([f"L{l}" for l in pv2.index])
    a2.set_xlabel("Head")
    a2.set_title("direkter Beitrag zum Ziel-Logit (W_O-Slice, letzte Position)", fontsize=9)
    fig.colorbar(im2, ax=a2, shrink=0.8)
    savefig("head_attribution")
    plt.show()


In [ ]:
# --- Kontrast 14: Steering am schlechtesten Head vs. am Residualstrom bei gleicher relativer Dosis --
with guarded("Head-Arm"):
    hs = table("I", "head_steering")
    hb = table("I", "head_steering_boot")
    std = hs[hs["kind"] == "standard"] if "kind" in hs else hs
    piv = std.groupby(["direction", "eps", "arm"])["B"].mean().unstack("arm")
    ax = piv.plot(kind="bar", figsize=(7, 3.8), width=0.7)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel("B [nats], Mittel über Varianten")
    ax.set_title("Head-Arm: Richtung durch den Head (W_O-Urbild) vs. Addition am Block-Ausgang", fontsize=9)
    ax.legend(fontsize=8)
    savefig("head_steering")
    plt.show()
    if len(hb):
        print("B_head − B_residual pro (Richtung, Dosis), Familien-Bootstrap:")
        show(hb)
    if len(std):
        print("Realisierte Richtung im Residualstrom (Kosinus) und Urbild-Fehler des Head-Arms:")
        show(std[std["arm"] == "head"].groupby(["direction", "eps"])[["rms_delta", "cos_realised_direction", "preimage_rel_err"]].mean())


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
